# 📈 Masterclass 01: Supervised Regression with L1/L2 Regularizations
This Jupyter Notebook provides a rigorous conceptual, mathematical, and practical breakdown of regularized linear models, culminating in a **dual-project** delivery:

1. **Project 1 (Theoretical Scratch)**: A from-scratch, highly vectorized implementation of Regularized Linear Regression using only `NumPy`.
2. **Project 2 (Applied Industry)**: A complete production pipeline targeting high-dimensional real-estate price forecasting with advanced feature scaling, collinearity filters (VIF), and grid search tuning.


## 📐 Part 1: Mathematical Foundations & LaTeX Proofs
In high-dimensional feature spaces, Ordinary Least Squares (OLS) suffers from high variance (overfitting) and numerical instability (multicollinearity). To mitigate this, we introduce penalization boundaries.

### 1. Ordinary Least Squares (OLS)
We seek to minimize the Mean Squared Error (MSE) loss function:
$$J(\theta) = \frac{1}{2m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)})^2 = \frac{1}{2m} (X\theta - y)^T (X\theta - y)$$
Taking the partial derivative with respect to $\theta$ and setting it to zero yields the **Normal Equations** closed-form solution:
$$\theta = (X^T X)^{-1} X^T y$$

### 2. Regularization Penalties
*   **Ridge Regression ($L_2$ Penalty)**: Adds a quadratic constraint to the weights: $J(\theta)_{Ridge} = J(\theta)_{OLS} + \frac{\lambda}{2m} \sum_{j=1}^{n} \theta_j^2$. This shrinks weights close to zero but keeps all features, solving multicollinearity.
*   **Lasso Regression ($L_1$ Penalty)**: Adds an absolute value constraint: $J(\theta)_{Lasso} = J(\theta)_{OLS} + \frac{\lambda}{m} \sum_{j=1}^{n} |\theta_j|$. This shrinks weights exactly to zero, performing automatic feature selection.
*   **Elastic Net**: Combines both L1 and L2 constraints with a mixing ratio $\alpha$:


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Generate synthetic non-linear polynomial data
np.random.seed(42)
X_raw = np.sort(5 * np.random.rand(80, 1), axis=0)
y_raw = np.sin(X_raw).ravel() + np.random.normal(0, 0.1, X_raw.shape[0])


## 🧠 Project 1: From-Scratch Regularized Estimator
Here we implement a complete gradient descent-driven estimator supporting regularized weights.


In [ ]:
import numpy as np

class RegularizedLinearRegression:
    def __init__(self, lr=0.01, epochs=1000, alpha=0.1, l1_ratio=0.5):
        self.lr = lr
        self.epochs = epochs
        self.alpha = alpha  # Lambda parameter
        self.l1_ratio = l1_ratio # 1.0 = Lasso, 0.0 = Ridge
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0.0

        for epoch in range(self.epochs):
            y_pred = np.dot(X, self.w) + self.b
            error = y_pred - y

            dw = (1 / n_samples) * np.dot(X.T, error)
            db = (1 / n_samples) * np.sum(error)

            l1_penalty = self.alpha * self.l1_ratio * np.sign(self.w)
            l2_penalty = self.alpha * (1 - self.l1_ratio) * self.w
            dw += l1_penalty + l2_penalty

            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict(self, X):
        return np.dot(X, self.w) + self.b


## 🧪 Project 2: Advanced Applied Real-Estate Valuation Pipeline
Below we build a production pipeline using `Scikit-Learn` to predict home prices, integrating Box-Cox scaling and multicollinearity tests.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet

# Create synthetic Housing DataFrame
data = pd.DataFrame({
    'sqft': np.random.normal(2000, 500, 200),
    'bedrooms': np.random.randint(2, 6, 200),
    'bathrooms': np.random.randint(1, 4, 200),
    'city_zone': np.random.choice(['Central', 'Suburbs', 'North'], 200),
    'price': np.random.normal(400000, 100000, 200)
})

X = data.drop(columns=['price'])
y = data['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['sqft', 'bedrooms', 'bathrooms']),
        ('cat', OneHotEncoder(), ['city_zone'])
    ]
)

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', ElasticNet(max_iter=5000))
])
print('Pipeline structure compiled successfully!')
